# 02 Spark 分析与导出 JSON

使用 PySpark 对灾害数据进行聚合，输出前端图表数据。

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, coalesce, lit, count, sum as spark_sum, month, year
import json
from pathlib import Path

In [ ]:
spark = (SparkSession.builder.appName('DisasterAnalytics').master('spark://spark-master:7077').getOrCreate())

df = (spark.read.option('header', True).csv('/home/jovyan/data/csv/disaster_2016_2020.csv')
      .withColumn('DirectEconomicLosses', coalesce(col('DirectEconomicLosses').cast('double'), lit(0.0)))
      .withColumn('DeathsNumber', coalesce(col('DeathsNumber').cast('double'), lit(0.0)))
      .withColumn('AffectedPopulation', coalesce(col('AffectedPopulation').cast('double'), lit(0.0)))
      .withColumn('CropsAffectedArea', coalesce(col('CropsAffectedArea').cast('double'), lit(0.0)))
      .withColumn('HouseCollapse', coalesce(col('HouseCollapse').cast('double'), lit(0.0)))
      .withColumn('SeriousDamage', coalesce(col('SeriousDamage').cast('double'), lit(0.0)))
      .withColumn('SecondaryDamage', coalesce(col('SecondaryDamage').cast('double'), lit(0.0)))
      .withColumn('MinorDamage', coalesce(col('MinorDamage').cast('double'), lit(0.0)))
      .withColumn('event_year', year(col('DeclareDate')))
      .withColumn('event_month', month(col('DeclareDate'))))

df.printSchema()

In [ ]:
event_classify = [{'name': r['EventClassify'] or '未知', 'value': int(r['value'])} for r in df.groupBy('EventClassify').agg(count('*').alias('value')).orderBy(col('value').desc()).collect()]

year_trend = [{'year': int(r['event_year']) if r['event_year'] else 0, 'deaths': float(r['deaths'] or 0), 'affected': float(r['affected'] or 0), 'loss': float(r['loss'] or 0)} for r in df.groupBy('event_year').agg(spark_sum('DeathsNumber').alias('deaths'), spark_sum('AffectedPopulation').alias('affected'), spark_sum('DirectEconomicLosses').alias('loss')).orderBy('event_year').collect()]

loss_top10 = [{'name': r['Province'] or '未知', 'value': float(r['loss'] or 0)} for r in df.groupBy('Province').agg(spark_sum('DirectEconomicLosses').alias('loss')).orderBy(col('loss').desc()).limit(10).collect()]

month_heat = [{'month': int(r['event_month'] or 0), 'count': int(r['cnt'])} for r in df.groupBy('event_month').agg(count('*').alias('cnt')).orderBy('event_month').collect()]

In [ ]:
kpi = {'totalEvents': int(df.count()), 'totalDeaths': int(df.agg(spark_sum('DeathsNumber').alias('x')).collect()[0]['x'] or 0), 'totalLossWanYuan': float(df.agg(spark_sum('DirectEconomicLosses').alias('x')).collect()[0]['x'] or 0), 'totalAffected': int(df.agg(spark_sum('AffectedPopulation').alias('x')).collect()[0]['x'] or 0)}

output = {'kpi': kpi, 'charts': {'eventClassify': event_classify, 'provinceHeat': loss_top10, 'yearTrend': year_trend, 'lossTop10': loss_top10, 'monthHeat': month_heat, 'populationVsLoss': [], 'houseDamageStack': [], 'cropTrend': year_trend, 'casualtyRose': []}}

out_file = Path('/home/jovyan/work/..') / 'backend' / 'output' / 'analysis_output.json'
out_file.parent.mkdir(parents=True, exist_ok=True)
out_file.write_text(json.dumps(output, ensure_ascii=False, indent=2), encoding='utf-8')
print('写入:', out_file)